# 02 — Recursion

## Why This Matters for DSA
Recursion is the natural language of trees, graphs, divide-and-conquer, and backtracking — nearly every notebook after this one leans on it directly. It's also where a lot of the complexity intuition from the previous notebook gets tested for real: a recursive function's shape (one call vs many, work before vs after the call) determines whether it's O(n) or O(2ⁿ), and whether it needs O(1) or O(n) stack space. This notebook builds the mental model precisely enough that you can look at a new recursive function and immediately see its cost, instead of just trusting that it "looks recursive so it's probably fine."

## Prerequisites
- `01_Complexity_Analysis` — this notebook constantly refers back to time/space complexity classes and the aggregate/call-count reasoning built there

## Learning Objectives
By the end of this notebook, you will be able to:
- Identify the base case and recursive case in any recursive function, and explain why both are required
- Trace a recursive call stack by hand, including the order calls happen and the order they return
- Distinguish linear recursion (one call per invocation) from tree recursion (multiple calls per invocation), and connect that shape directly to O(n) vs O(2ⁿ)
- Explain tail recursion, why C++ doesn't guarantee it gets optimized, and when that matters
- Recognize the include/exclude recursive pattern used for subsets, and see how it's structurally identical to tree-recursive Fibonacci
- Apply memoization to convert exponential recursion into linear recursion
- Convert a recursive function into an iterative one, including the explicit-stack technique for non-tail recursion

## Checklist Before Moving On
- [ ] I can write a recursive function's base case first, every time, before writing the recursive case
- [ ] I can predict whether a recursive function's call count is O(n) or O(2ⁿ) just from how many times it calls itself
- [ ] I know C++ does not guarantee tail-call optimization, and what that means practically
- [ ] I can convert an "include/exclude" recursive subset generator into a memoized or iterative form if needed
- [ ] I understand memoization turns overlapping recursive calls into a linear number of unique calls
- [ ] I can explain when to prefer an iterative solution over a recursive one purely for stack-depth reasons

## Table of Contents
1. Anatomy of a Recursive Function — Base Case, Recursive Case, and the Call Stack
2. Linear Recursion vs Tree Recursion
3. Tail Recursion vs Head Recursion
4. Recursion on Arrays and Strings
5. Multiple Recursive Calls — The Include/Exclude Pattern
6. Memoization — Turning Exponential Recursion into Linear
7. Converting Recursion to Iteration
8. Phase Checkpoint, Cheat Sheet, and Answer Key


## 1. Anatomy of a Recursive Function — Base Case, Recursive Case, and the Call Stack

### The Why
Every recursion bug that isn't a logic error is usually a MISSING or WRONG base case — either it never gets hit (infinite recursion, stack overflow) or it gets hit at the wrong point (off-by-one results). Getting comfortable tracing the call stack by hand is what lets you debug these confidently instead of guessing.

### Core Mechanism
- **Base case:** the condition that stops recursion and returns directly, without calling itself again. Every recursive function needs at least one.
- **Recursive case:** calls the function again on a SMALLER or SIMPLER version of the problem, moving measurably closer to the base case each time.
- **The call stack:** each recursive call is a new **stack frame** — its own local variables and a note of where to resume once it returns. Frames stack up as calls go deeper, then unwind in exact reverse order as each call returns and is popped off.
- The traced example below prints on the way DOWN (before recursing) and on the way UP (after the recursive call returns) — watch how the indentation grows before any multiplication happens, and how results come back in the exact reverse order the calls were made.

### Common Pitfalls
- Writing the recursive case first and forgetting the base case entirely, or writing a base case that's never actually reachable from the input (e.g. checking `n == 0` when `n` could skip past 0, like decrementing by 2 from an odd start).
- Recursing on a version of the problem that ISN'T smaller — this causes infinite recursion just as surely as a missing base case.


In [ ]:
%%writefile temp.cpp
#include <iostream>
using namespace std;

// Every recursive function needs exactly two things:
// 1. A BASE CASE -- a condition that stops the recursion, returned directly, no further calls
// 2. A RECURSIVE CASE -- calls itself on a SMALLER version of the problem, moving toward the base case

int factorial(int n) {
    if (n <= 1) return 1;               // base case: stops the recursion
    return n * factorial(n - 1);        // recursive case: smaller problem (n-1), moves toward base case
}

// tracing WITH indentation to visualize the call stack depth as it grows and shrinks
int factorialTraced(int n, int depth = 0) {
    string indent(depth * 2, ' ');
    cout << indent << "-> factorialTraced(" << n << ") called\n";

    if (n <= 1) {
        cout << indent << "<- base case hit, returning 1\n";
        return 1;
    }

    int result = n * factorialTraced(n - 1, depth + 1);
    cout << indent << "<- factorialTraced(" << n << ") returning " << result << "\n";
    return result;
}

int main() {
    cout << "factorial(5) = " << factorial(5) << "\n\n";

    cout << "Traced call stack for factorialTraced(4):\n";
    factorialTraced(4);
    // notice the pattern: calls go DOWN (n=4,3,2,1) before anything returns, then
    // returns come back UP in reverse order (1,2,3,4) -- this is the call stack:
    // each call waits, stacked on top of the next, until the base case unwinds them all

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 2. Linear Recursion vs Tree Recursion

### The Why
This is the single most useful classification for predicting a recursive function's complexity at a glance, before you've traced anything by hand: count how many times the function calls ITSELF inside its own body.

### Core Mechanism
- **Linear recursion:** exactly ONE recursive call per invocation. The call structure is a straight chain — call depth n produces n calls total, and O(n) stack depth. `power()` below is linear: it calls itself exactly once.
- **Tree recursion:** MORE THAN ONE recursive call per invocation. The call structure branches into a tree rather than a line. Naive Fibonacci is the canonical example — two calls per invocation means the call tree's node count grows exponentially with depth, which is the structural reason naive Fibonacci is O(2ⁿ) (as measured empirically in `01_Complexity_Analysis`).
- This is a DIRECT visual/structural explanation for a fact you already saw empirically: tree recursion with branching factor 2 roughly doubles total calls for every +1 to the input, because each new level of the tree has roughly twice as many nodes as the level above it.

### Common Pitfalls
- Assuming any recursive function is "probably O(n)" without checking how many times it calls itself — one call vs two calls is the difference between fast and unusable at moderate input sizes.


In [ ]:
%%writefile temp.cpp
#include <iostream>
using namespace std;

// LINEAR recursion: each call makes exactly ONE recursive call.
// The call "chain" is a straight line -- O(n) calls total, O(n) stack depth.
long long power(int base, int exp) {
    if (exp == 0) return 1;
    return base * power(base, exp - 1);   // exactly one recursive call per invocation
}

// TREE recursion: each call makes MORE THAN ONE recursive call.
// The call structure branches into a TREE, not a line -- this is what makes naive
// Fibonacci exponential: the tree's node count grows exponentially with depth.
long long fibCallCount = 0;
long long fib(int n, int depth = 0, int maxDepthSeen = 0) {
    fibCallCount++;
    if (n <= 1) return n;
    return fib(n - 1, depth + 1) + fib(n - 2, depth + 1);   // TWO recursive calls per invocation
}

int main() {
    cout << "power(2, 10) = " << power(2, 10) << "\n";

    for (int n : {5, 10, 15}) {
        fibCallCount = 0;
        long long result = fib(n);
        cout << "fib(" << n << ")=" << result << ", total calls=" << fibCallCount << "\n";
    }
    // linear recursion (power): n calls for input n -- a straight chain
    // tree recursion (fib): calls roughly DOUBLE every time n increases by 1 --
    // this is exactly the O(2^n) blowup from the complexity analysis notebook,
    // and now you can see structurally WHY: every call spawns 2 more calls,
    // so the call tree has roughly 2^n total nodes by depth n

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 3. Tail Recursion vs Head Recursion

### The Why
"Just make it tail-recursive" is common advice for saving stack space — but in C++, that advice needs a caveat almost nobody mentions: the language doesn't guarantee the optimization actually happens.

### Core Mechanism
- **Head recursion** (the work happens AFTER the recursive call returns): `factorialHead` must wait for `factorialHead(n-1)` to fully return before it can multiply by `n`. Every pending call sits on the stack, waiting, until the base case is hit and unwinding begins — O(n) stack frames alive at once.
- **Tail recursion** (the recursive call is the LAST thing that happens, nothing left to do after): `factorialTail` threads the running result through an accumulator parameter, so the recursive call's return value IS the final answer directly — nothing left to compute afterward.
- **The theoretical benefit:** in a language that GUARANTEES tail-call optimization, a tail-recursive call reuses the same stack frame instead of pushing a new one — turning O(n) space into O(1) space, since there's nothing left to remember once you make the tail call.
- **The C++ reality:** C++ does NOT guarantee tail-call optimization. It's an optional compiler optimization, typically only applied at `-O2` or higher, and only when the compiler can prove it's safe — it is NOT a language guarantee you can depend on. Writing a function in tail-recursive style is good practice (it's often clearer), but you should never assume it eliminates stack usage in C++ the way it would in a language like Scheme that mandates the optimization.

### Common Pitfalls
- Relying on tail-call optimization to handle very deep recursion (say, 10⁶+ levels) in C++ — without a compiler guarantee, this can still stack-overflow. Use an explicit loop (Section 7) if depth could be large and stack safety actually matters.


In [ ]:
%%writefile temp.cpp
#include <iostream>
using namespace std;

// HEAD recursion (a misleading name -- really means "work happens AFTER the recursive call"):
// you must return from the deepest call before any multiplication happens.
// Every pending call sits on the stack, waiting, until the base case is reached.
long long factorialHead(int n) {
    if (n <= 1) return 1;
    return n * factorialHead(n - 1);   // the multiplication happens AFTER the recursive call returns
}

// TAIL recursion: the recursive call is the VERY LAST thing that happens -- nothing left
// to do after it returns. The "work so far" is threaded through as an accumulator parameter
// instead of being computed on the way back up.
long long factorialTail(int n, long long accumulator = 1) {
    if (n <= 1) return accumulator;
    return factorialTail(n - 1, n * accumulator);   // nothing happens after this call returns --
                                                       // its result IS the final result, directly
}

int main() {
    cout << "factorialHead(10) = " << factorialHead(10) << "\n";
    cout << "factorialTail(10) = " << factorialTail(10) << "\n";

    // WHY THIS MATTERS: in languages that GUARANTEE tail-call optimization (TCO), a tail-recursive
    // function reuses the SAME stack frame for every call, running in O(1) space instead of O(n).
    // C++ does NOT guarantee TCO -- it's an optional optimization a compiler MAY apply, typically
    // only at -O2 or higher, and only in simple enough cases. NEVER rely on C++ eliminating your
    // recursion's stack usage just because you wrote it in tail form -- for genuinely large n,
    // prefer an explicit loop if stack depth is a real concern (see Section 9).

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 4. Recursion on Arrays and Strings

### The Why
This is the pattern you'll use constantly once trees and graphs show up: process one element, then recurse on "everything else." Getting comfortable with it on plain arrays and strings first — where there's no pointer/node complexity to distract you — makes the tree/graph versions much easier later.

### Core Mechanism
- **Sum an array recursively:** base case is an empty remaining range (index has walked off the end); recursive case adds the current element to the sum of everything after it.
- **Reverse a string recursively:** base case is an empty string (reverses to itself); recursive case puts the LAST character first, then recurses on everything before it.
- **Palindrome check recursively:** base case is the two pointers crossing or meeting (everything's been checked); recursive case compares the outer two characters, then recurses INWARD on the pair just inside them.
- The common shape across all three: shrink the problem by one element (or one pair) per call, and trust the recursive call to correctly handle "the rest."

### Common Pitfalls
- Writing `reverseString` with `s.substr(0, s.size() - 1)` as shown works but creates a NEW string copy at every call — O(n) work per call, O(n) calls, so this specific implementation is O(n²) despite reversing being conceptually an O(n) operation. This is a good example of recursion elegance costing real performance; an index-based, in-place version avoids the repeated copying.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <string>
using namespace std;

// recursive sum over an array: base case is an empty range, recursive case handles
// one element and delegates the rest
int sumArray(const vector<int>& v, int idx) {
    if (idx == (int)v.size()) return 0;      // base case: nothing left to add
    return v[idx] + sumArray(v, idx + 1);    // handle one element, recurse on the rest
}

// recursive string reversal: build the reversed string by putting the LAST character
// first, then recursing on everything before it
string reverseString(const string& s) {
    if (s.empty()) return s;                          // base case: empty string reverses to itself
    return s.back() + reverseString(s.substr(0, s.size() - 1));
}

// recursive palindrome check: strip matching outer characters one pair at a time
bool isPalindrome(const string& s, int left, int right) {
    if (left >= right) return true;         // base case: pointers crossed or met -- checked everything
    if (s[left] != s[right]) return false;  // mismatch found -- not a palindrome
    return isPalindrome(s, left + 1, right - 1);   // strip the outer pair, recurse inward
}

int main() {
    vector<int> nums{1, 2, 3, 4, 5};
    cout << "sumArray = " << sumArray(nums, 0) << "\n";

    cout << "reverseString(\"hello\") = " << reverseString("hello") << "\n";

    cout << "isPalindrome(\"racecar\") = " << isPalindrome("racecar", 0, 6) << "\n";
    cout << "isPalindrome(\"hello\") = " << isPalindrome("hello", 0, 4) << "\n";

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 5. Multiple Recursive Calls — The Include/Exclude Pattern

### The Why
This is the pattern behind subset generation, and it's structurally IDENTICAL to tree-recursive Fibonacci from Section 2 — two recursive calls per invocation, one call tree. Recognizing it here is what makes backtracking (the next notebook) feel like a natural extension rather than a brand new topic.

### Core Mechanism
- For each element in the input, you make exactly two choices: **include it** in the current subset, or **exclude it**. Each choice is a recursive call — two calls per invocation, same branching shape as tree-recursive Fibonacci.
- The recursion bottoms out (base case) once every element has had a decision made about it — at that point, the current partial subset is a complete, valid subset, and gets recorded.
- **Backtracking is visible here already:** after the "include" branch pushes an element onto the current subset and recurses, it must `pop_back()` that element before returning — undoing the choice so the NEXT sibling call (a different decision at a different point in the tree) starts from a clean state. This undo-before-returning step is the defining feature of backtracking, and you're already doing it.
- For `n` elements, there are `2ⁿ` possible subsets, and the recursion makes roughly `2ⁿ⁺¹` total calls to produce them — the exact same "doubles per added element" growth pattern as naive Fibonacci, just applied to decision-making instead of arithmetic.

### Common Pitfalls
- Forgetting the `pop_back()` (or equivalent undo step) after the "include" recursive call returns — without it, later branches would incorrectly still have earlier elements in their "current subset," corrupting every subset generated after the first bug.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// generating all subsets of a set is the classic "multiple recursive calls" pattern:
// for EACH element, you either INCLUDE it or EXCLUDE it -- two choices, two recursive calls,
// exactly the same branching shape as tree-recursive Fibonacci, just applied to decision-making
// instead of arithmetic. This pattern is the direct bridge into backtracking (next notebook).
long long subsetCalls = 0;

void generateSubsets(const vector<int>& nums, int idx, vector<int>& current, vector<vector<int>>& result) {
    subsetCalls++;
    if (idx == (int)nums.size()) {
        result.push_back(current);   // reached the end of decisions -- record this subset
        return;
    }

    // choice 1: EXCLUDE nums[idx]
    generateSubsets(nums, idx + 1, current, result);

    // choice 2: INCLUDE nums[idx]
    current.push_back(nums[idx]);
    generateSubsets(nums, idx + 1, current, result);
    current.pop_back();   // backtrack: undo the choice before returning to the caller
}

int main() {
    vector<int> nums{1, 2, 3};
    vector<int> current;
    vector<vector<int>> result;
    generateSubsets(nums, 0, current, result);

    cout << "subsets of {1,2,3}:\n";
    for (auto &subset : result) {
        cout << "{ ";
        for (int x : subset) cout << x << " ";
        cout << "}\n";
    }
    cout << "total subsets=" << result.size() << " (2^3=8), total calls=" << subsetCalls << "\n";

    // for n elements, there are 2^n possible subsets -- and generateSubsets makes roughly
    // 2^(n+1) calls to produce them, because it's branching into exactly 2 choices per element,
    // same growth shape as tree-recursive fibonacci, just guaranteed exact rather than approximate

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 6. Memoization — Turning Exponential Recursion into Linear

### The Why
Tree recursion is expensive because of REDUNDANT work — the same sub-problem gets recomputed from scratch every time it's reached via a different path through the call tree. Memoization is the fix: remember an answer the first time it's computed, and reuse it instead of recomputing.

### Core Mechanism
- **The redundancy:** `fib(5)` calls `fib(3)` (via the `fib(4)` branch) — but also independently via nothing directly, though `fib(3)` gets requested from multiple places as `n` grows; the deeper the tree, the more often the SAME `n` gets asked for again from a different branch.
- **The fix:** keep a lookup table (`memo`) indexed by `n`. Before computing `fib(n)` recursively, check if it's already in the table — if so, return the cached value immediately with NO further recursion. Otherwise compute it normally, then store the result before returning.
- **The result:** every distinct value of `n` from `0` to the target gets computed exactly ONCE. Total calls become linear — O(n) — instead of exponential. This technique is called **top-down dynamic programming**: same recursive structure as the naive version, with a cache layered on top.
- The measured call counts make this concrete: naive `fib(40)` takes over 300 million calls; memoized `fib(40)` takes 79 — exactly `2n - 1` for `n=40`.

### Common Pitfalls
- Initializing the memo table with a sentinel value that could collide with a REAL result (e.g. using `0` as "not yet computed" when `0` is also a legitimate answer for some input) — use a value that's genuinely impossible as an answer, like `-1` for a problem where results are always non-negative.
- Forgetting to actually WRITE to the memo table after computing a value — this defeats the entire purpose; the function will still be correct but back to exponential, since nothing ever gets reused.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// naive recursive fibonacci: recomputes the SAME sub-problems over and over.
// fib(5) calls fib(3) TWICE, fib(2) THREE times, etc -- massive redundant work.
long long naiveCalls = 0;
long long fibNaive(int n) {
    naiveCalls++;
    if (n <= 1) return n;
    return fibNaive(n - 1) + fibNaive(n - 2);
}

// memoized fibonacci: cache each result the FIRST time it's computed, and reuse it
// on every subsequent request for the same n. This is "top-down dynamic programming" --
// same recursive structure, but with a lookup table preventing redundant work.
long long memoCalls = 0;
long long fibMemo(int n, vector<long long>& memo) {
    memoCalls++;
    if (n <= 1) return n;
    if (memo[n] != -1) return memo[n];         // already computed -- reuse it, no further recursion
    memo[n] = fibMemo(n - 1, memo) + fibMemo(n - 2, memo);
    return memo[n];
}

int main() {
    for (int n : {20, 30, 40}) {
        naiveCalls = 0;
        long long naiveResult = fibNaive(n);

        memoCalls = 0;
        vector<long long> memo(n + 1, -1);
        long long memoResult = fibMemo(n, memo);

        cout << "n=" << n << " naive calls=" << naiveCalls
             << " memo calls=" << memoCalls
             << " (both correctly compute " << naiveResult << "==" << memoResult << ")\n";
    }
    // naive calls grow exponentially (roughly doubling per +1 to n) -- see notebook 01
    // memo calls grow LINEARLY: exactly 2n-1 calls, because each n from 0 to the target
    // is computed exactly once, memoization turns O(2^n) into O(n)

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 7. Converting Recursion to Iteration

### The Why
Sometimes recursion is the wrong tool for production code — not because it's conceptually wrong, but because deep recursion can exhaust the call stack (typically after tens of thousands of frames, though the exact number depends on frame size and platform), while an equivalent loop uses a fixed, small amount of stack space regardless of "how deep" the logical recursion would have gone.

### Core Mechanism
- **Tail-recursive functions convert to loops almost mechanically:** the accumulator parameter becomes a regular loop variable, and each recursive call becomes one loop iteration. `factorialTail` from Section 3 becomes `factorialIterative` with essentially no conceptual translation needed.
- **Non-tail recursion needs an EXPLICIT STACK** to replace the work the implicit call stack was doing. The general technique: create your own `stack` of "frames," where each frame tracks which "step" of the original recursive logic it's currently on (e.g. "about to compute the first sub-call," "first sub-call done, about to compute the second," "both done, ready to combine and return"). This is more code, but it's a MECHANICAL translation of what the call stack was already doing implicitly — nothing conceptually new, just made visible and under your control.
- The explicit-stack `fib` example is deliberately verbose to show the technique clearly — in practice, you'd almost always prefer memoization (Section 6) over this kind of manual stack simulation for Fibonacci specifically. The technique matters more for cases where you genuinely need iteration (stack-depth safety) AND can't easily restructure the algorithm another way.

### Common Pitfalls
- Converting head/tail recursion to iteration correctly, but botching a TREE recursion conversion by forgetting to save intermediate results (like the `left` value in the traced example) before making the "second" recursive call — without explicitly saving it, it gets overwritten before you can combine both branches.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <stack>
#include <vector>
using namespace std;

// tail-recursive-style factorial converted DIRECTLY to a loop: this conversion is
// mechanical when the recursion is tail-recursive -- the accumulator parameter becomes
// a regular loop variable, and the recursive call becomes a loop iteration.
long long factorialIterative(int n) {
    long long accumulator = 1;
    for (int i = n; i > 1; i--) {
        accumulator *= i;
    }
    return accumulator;
}

// when recursion ISN'T tail-recursive (like tree recursion, or head recursion with
// work after the call), converting to iteration needs an EXPLICIT STACK to replace
// the implicit call stack the recursion was using. Here's naive fibonacci's call
// pattern simulated with an explicit stack instead of language-level recursion.
// (this specific example uses a stack of "pending work" -- not a performance win over
// memoization, but demonstrates the general recursion-to-iteration technique)
long long fibIterativeStack(int n) {
    if (n <= 1) return n;

    // simulate the call stack: each stack entry is (n, state) where state tracks
    // whether we're about to compute fib(n-1), fib(n-2), or ready to combine both
    struct Frame { int n; int state; long long left; };
    stack<Frame> st;
    st.push({n, 0, 0});
    long long lastReturn = 0;

    while (!st.empty()) {
        Frame &top = st.top();
        if (top.n <= 1) {
            lastReturn = top.n;
            st.pop();
        } else if (top.state == 0) {
            top.state = 1;
            st.push({top.n - 1, 0, 0});   // "recurse" into fib(n-1)
        } else if (top.state == 1) {
            top.left = lastReturn;         // save fib(n-1)'s result
            top.state = 2;
            st.push({top.n - 2, 0, 0});   // "recurse" into fib(n-2)
        } else {
            lastReturn = top.left + lastReturn;   // combine both results
            st.pop();
        }
    }
    return lastReturn;
}

int main() {
    cout << "factorialIterative(10) = " << factorialIterative(10) << "\n";
    cout << "fibIterativeStack(15) = " << fibIterativeStack(15) << "\n";

    // WHEN TO CONVERT: prefer iteration when recursion depth could exceed the stack size
    // (typically a few tens of thousands of frames before overflow, though the exact limit
    // depends on frame size and platform) -- e.g. recursing once per element of a very large
    // input. Tail-recursive logic converts to a plain loop trivially. Non-tail recursion
    // converts by introducing an explicit stack that mirrors what the call stack was doing.

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 8. Phase Checkpoint, Cheat Sheet, and Answer Key

### Recursion Pattern Cheat Sheet

| Pattern | Calls per invocation | Typical complexity | Example |
|---|---|---|---|
| Linear recursion | 1 | O(n) time, O(n) space | Factorial, array sum, power |
| Tree recursion (unmemoized) | 2+ | O(2ⁿ) time, O(n) space (depth, not node count) | Naive Fibonacci |
| Tree recursion (memoized) | 2+, but cached | O(n) time, O(n) space | Memoized Fibonacci |
| Include/exclude | 2 | O(2ⁿ) time (2ⁿ subsets to enumerate) | Subset generation, subset sum |
| Tail recursion | 1, as the last operation | Same time as head recursion; O(1) space ONLY if compiler optimizes it (not guaranteed in C++) | Accumulator-style factorial |

### Checkpoint Questions
1. What are the two required components of every recursive function, and what happens if either is missing or wrong?
2. A recursive function calls itself twice per invocation. Without tracing it, what complexity class do you expect, and why?
3. Why doesn't "I wrote it as tail recursion" guarantee O(1) space in C++?
4. In the include/exclude subset pattern, why is the `pop_back()` after the "include" branch essential?
5. Explain, in one or two sentences, why memoization turns fib's complexity from O(2ⁿ) to O(n).
6. When would you deliberately convert a recursive solution to an iterative one, even though the recursive version is correct and easier to read?

### Answer Key
1. A base case (stops the recursion, returns directly) and a recursive case (calls itself on a smaller/simpler version of the problem, moving toward the base case). Missing or unreachable base case → infinite recursion → stack overflow. A recursive case that doesn't actually shrink the problem → same failure.
2. O(2ⁿ) — two calls per invocation means the call tree's node count roughly doubles with each added unit of input size, the same branching shape as naive Fibonacci.
3. C++ does not guarantee tail-call optimization — it's an optional compiler optimization applied only in some cases, typically at higher optimization levels, never a language-level guarantee. Writing tail-recursive style is good practice for clarity but doesn't itself change C++'s stack usage without the compiler choosing to optimize it.
4. Without undoing the "include" choice, later sibling recursive calls (different decisions at the same or earlier tree level) would incorrectly still have that earlier element present in the "current subset" they're building, corrupting every subsequent result.
5. Memoization ensures each distinct sub-problem (each value of n) is computed only once, with all repeat requests served from a cache instead of triggering fresh recursive branches — turning what was exponentially many REPEATED computations into linearly many DISTINCT ones.
6. When recursion depth could realistically approach or exceed the platform's stack size (very large inputs processed one-recursive-call-per-element), when a hard guarantee on space usage is needed rather than an optimization you're hoping the compiler applies, or when profiling shows function-call overhead itself is a measurable bottleneck.

### Mini Project
Implement `countWays(n)`: the number of distinct ways to climb `n` stairs, taking either 1 or 2 steps at a time (this recurrence is `countWays(n) = countWays(n-1) + countWays(n-2)`, structurally identical to Fibonacci).
1. Write the naive tree-recursive version first, and count its calls for a few increasing `n`, confirming the exponential blowup pattern from Section 2.
2. Add memoization and re-measure the call count, confirming it drops to linear.
3. Convert your memoized version to a fully iterative bottom-up version (no recursion at all, just a loop building up answers from `countWays(0)` and `countWays(1)`) and confirm it produces the same answers.

This exercises: base/recursive case design, tree-recursion complexity recognition, memoization, and the recursion-to-iteration conversion — the complete arc of this notebook.
